# LeetCode 2558 — Take Gifts From the Richest Pile bằng Python

Link đề: https://leetcode.com/problems/take-gifts-from-the-richest-pile/ · Độ khó: **Easy** · Chủ đề: Array, Heap (Priority Queue), Simulation

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def pickGifts(self, gifts: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu quy trình mô phỏng: mỗi giây chọn **đống quà nhiều nhất**, để lại `floor(sqrt(số quà))`, lấy phần còn lại; lặp `k` giây rồi tính tổng quà còn lại.
2. Biết dùng `math.isqrt` (căn bậc hai **nguyên** chính xác) và hiểu vì sao nên tránh `int(x ** 0.5)` khi muốn chắc chắn.
3. Hiểu **max-heap** bằng `heapq` với số đổi dấu, và `heapreplace` để "lấy đỉnh và đẩy giá trị mới" trong một thao tác.
4. Viết 4 cách: mô phỏng với `max` + `index`, max-heap bằng `heappop`/`heappush`, mảng sắp xếp với `bisect.insort`, và `heapreplace` kèm thoát sớm khi mọi đống đều bằng `1`; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: dùng `ceil` thay vì `floor`, lấy căn của mọi đống mỗi giây, quên đẩy lại heap, và chọn đống nhỏ nhất thay vì lớn nhất.

## 2. Đề bài & ví dụ

Cho mảng `gifts`, trong đó `gifts[i]` là số quà ở đống thứ `i`, và số nguyên `k`. Mỗi **giây**, bạn thực hiện:

1. Chọn đống có **nhiều quà nhất** (nếu nhiều đống bằng nhau, chọn đống nào cũng được).
2. Để lại `floor(sqrt(gifts[i]))` quà trong đống đó và **lấy đi** phần còn lại.

Trả về **tổng số quà còn lại** sau `k` giây.

| Ví dụ | gifts | k | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[25,64,9,4,100]` | `4` | `29` | `100->10`, `64->8`, `25->5`, `10->3`; còn `[5,8,9,4,3]`, tổng `29` |
| 2 | `[1,1,1,1]` | `4` | `4` | `sqrt(1) = 1`, không đống nào đổi; tổng `4` |

Constraints: `1 <= gifts.length <= 10^3`; `1 <= gifts[i] <= 10^9`; `1 <= k <= 10^3`.

In [1]:
import math
gifts, k = [25, 64, 9, 4, 100], 4
g = gifts[:]
for sec in range(1, k + 1):
    i = g.index(max(g))
    g[i] = math.isqrt(g[i])
    print(f"giây {sec}: chọn đống {i}, còn lại {g}")
print("tổng còn lại:", sum(g), "(mong đợi 29)")

giây 1: chọn đống 4, còn lại [25, 64, 9, 4, 10]
giây 2: chọn đống 1, còn lại [25, 8, 9, 4, 10]
giây 3: chọn đống 0, còn lại [5, 8, 9, 4, 10]
giây 4: chọn đống 4, còn lại [5, 8, 9, 4, 3]
tổng còn lại: 29 (mong đợi 29)


## 3. Kiến thức nền

### 3.1. `math.isqrt` — căn bậc hai nguyên

`math.isqrt(n)` trả về **phần nguyên** của căn bậc hai của `n` (số nguyên lớn nhất `r` sao cho `r * r <= n`), tính hoàn toàn bằng số nguyên nên **không có sai số** dấu phẩy động. `math.sqrt(n)` trả về số thực.

In [2]:
import math
for x in (1, 4, 10, 99, 100, 10**9):
    print(x, "-> isqrt:", math.isqrt(x), "| sqrt:", math.sqrt(x))

1 -> isqrt: 1 | sqrt: 1.0
4 -> isqrt: 2 | sqrt: 2.0
10 -> isqrt: 3 | sqrt: 3.1622776601683795
99 -> isqrt: 9 | sqrt: 9.9498743710662
100 -> isqrt: 10 | sqrt: 10.0
1000000000 -> isqrt: 31622 | sqrt: 31622.776601683792


### 3.2. Lấy đống nhiều nhất bằng `max` và `index`

`max(g)` trả giá trị lớn nhất; `g.index(v)` trả chỉ số **đầu tiên** của `v`. Cả hai tốn `O(n)`. Khi có nhiều đống bằng nhau, chọn đống nào cũng cho cùng tổng cuối vì chúng giống hệt nhau.

In [3]:
g = [25, 64, 9, 4, 100]
m = max(g)
i = g.index(m)
print(m, i)

100 4


### 3.3. `heapq` — min-heap và mẹo max-heap

`heapq` chỉ có **min-heap** (`heap[0]` là nhỏ nhất). Để lấy lớn nhất, ta lưu **số đối** `-x`: đỉnh của heap là `-max`. Các thao tác: `heapify(list)` (`O(n)`), `heappop(h)` (`O(log n)`), `heappush(h, x)` (`O(log n)`).

In [4]:
import heapq
h = [-x for x in [25, 64, 9, 4, 100]]
heapq.heapify(h)
print("đỉnh heap:", h[0], "-> giá trị lớn nhất:", -h[0])
top = -heapq.heappop(h)
print("lấy ra:", top, "| còn", len(h), "phần tử")
heapq.heappush(h, -math.isqrt(top))
print("đẩy lại sqrt:", -h[0])

đỉnh heap: -100 -> giá trị lớn nhất: 100
lấy ra: 100 | còn 4 phần tử
đẩy lại sqrt: 64


### 3.4. `heapq.heapreplace` — pop rồi push trong một bước

`heapq.heapreplace(h, item)` lấy phần tử nhỏ nhất ra và đẩy `item` vào, **hiệu quả hơn** `heappop` + `heappush` (chỉ một lần sắp xếp lại). Lưu ý: nó pop **trước** rồi mới push.

In [5]:
h = [-x for x in [25, 64, 9, 4, 100]]
heapq.heapify(h)
old = heapq.heapreplace(h, -math.isqrt(-h[0]))      # thay đỉnh bằng giá trị mới
print("đã lấy ra:", -old, "| đỉnh mới:", -h[0])

đã lấy ra: 100 | đỉnh mới: 64


### 3.5. `bisect.insort` — chèn vào mảng đã sắp xếp

`bisect.insort(a, x)` chèn `x` vào mảng `a` đã sắp xếp (tăng dần) sao cho `a` vẫn sắp xếp; tốn `O(n)` vì phải dịch phần tử. Khi mảng tăng dần, phần tử lớn nhất là `a[-1]` và `a.pop()` lấy nó trong `O(1)`.

In [6]:
from bisect import insort
a = sorted([25, 64, 9, 4, 100])
print(a, "| lớn nhất:", a[-1])
top = a.pop()
insort(a, math.isqrt(top))
print(a)

[4, 9, 25, 64, 100] | lớn nhất: 100
[4, 9, 10, 25, 64]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay `gifts = [25, 64, 9, 4, 100]`, `k = 4`:

| Giây | Lớn nhất | Sau khi lấy | Mảng |
|---|---|---|---|
| 1 | 100 | `isqrt(100) = 10` | `[25,64,9,4,10]` |
| 2 | 64 | `isqrt(64) = 8` | `[25,8,9,4,10]` |
| 3 | 25 | `isqrt(25) = 5` | `[5,8,9,4,10]` |
| 4 | 10 | `isqrt(10) = 3` | `[5,8,9,4,3]` |

Tổng `5 + 8 + 9 + 4 + 3 = 29`.

**Nhận xét:** sau mỗi giây, đống bị chọn **giảm rất mạnh** (căn bậc hai). Một số tối đa `10^9` sau 5 lần lấy căn còn `1` (`10^9 -> 31622 -> 177 -> 13 -> 3 -> 1`). Khi mọi đống đều là `1` thì mô phỏng không còn thay đổi gì nữa.

In [7]:
x = 10**9
chain = [x]
while chain[-1] > 1:
    chain.append(math.isqrt(chain[-1]))
print(chain, "-> sau", len(chain) - 1, "lần lấy căn thì về 1")

[1000000000, 31622, 177, 13, 3, 1] -> sau 5 lần lấy căn thì về 1


## 5. Cách 1 — Brute Force / baseline: `max` + `index` mỗi giây

**Ý tưởng:** làm đúng như đề. Mỗi giây tìm giá trị lớn nhất `m = max(g)`, tìm chỉ số `i = g.index(m)`, gán `g[i] = isqrt(m)`. Cuối cùng trả `sum(g)`. Ta làm trên **bản sao** của `gifts`.

In [8]:
import math
from typing import List

class SolutionBrute:
    def pickGifts(self, gifts: List[int], k: int) -> int:
        g = gifts[:]                                  # không phá đầu vào
        for _ in range(k):
            i = g.index(max(g))
            g[i] = math.isqrt(g[i])
        return sum(g)

In [9]:
sol = SolutionBrute()
print(sol.pickGifts([25, 64, 9, 4, 100], 4))     # 29
print(sol.pickGifts([1, 1, 1, 1], 4))            # 4

29
4


**Độ phức tạp:** Time `O(k * n)` (mỗi giây `max` và `index` đều `O(n)`); với `k, n <= 10^3` là `10^6` thao tác, vẫn rất nhanh. Space `O(n)` cho bản sao. **Nút thắt:** tìm lại lớn nhất bằng quét toàn mảng mỗi giây; heap cho phép lấy lớn nhất trong `O(log n)`.

## 6. Ý tưởng thuật toán / bước đột phá

Ta cần liên tục **lấy phần tử lớn nhất rồi đưa giá trị mới (nhỏ hơn) trở lại**: đây là bài toán điển hình của **hàng đợi ưu tiên (heap)**.

- Dựng max-heap (lưu số đối) từ `gifts` trong `O(n)`.
- Lặp `k` lần: lấy đỉnh `top`, đẩy lại `isqrt(top)`.
- Cuối cùng cộng các phần tử còn lại (nhớ đổi dấu).

Thêm một **tối ưu**: nếu đỉnh heap bằng `1` thì mọi đống đều `1` (vì đỉnh là lớn nhất) và `isqrt(1) = 1` nên các giây sau không đổi gì; thoát sớm.

**Quy tắc:** *luôn lấy đống lớn nhất bằng heap; đống bị chọn chuyển thành `isqrt(giá trị)`.*

**Invariant:** sau `t` giây, heap chứa đúng các giá trị của mô phỏng ở thời điểm `t` (dưới dạng số đối).

## 7. Các cách tiếp theo

### Cách 2 — Max-heap bằng `heappop` + `heappush`

In [10]:
import heapq
import math
from typing import List

class SolutionHeap:
    def pickGifts(self, gifts: List[int], k: int) -> int:
        h = [-x for x in gifts]
        heapq.heapify(h)
        for _ in range(k):
            top = -heapq.heappop(h)
            heapq.heappush(h, -math.isqrt(top))
        return -sum(h)

In [11]:
sol = SolutionHeap()
print(sol.pickGifts([25, 64, 9, 4, 100], 4))     # 29
print(sol.pickGifts([1, 1, 1, 1], 4))            # 4

29
4


**Độ phức tạp:** Time `O(n + k log n)` (`heapify` `O(n)`, mỗi giây `O(log n)`), Space `O(n)`.

### Cách 3 — Mảng sắp xếp + `insort`

Giữ mảng `a` sắp xếp tăng; lớn nhất là `a[-1]`. Mỗi giây `pop()` lớn nhất và `insort` giá trị mới vào đúng chỗ.

In [12]:
import math
from bisect import insort
from typing import List

class SolutionSorted:
    def pickGifts(self, gifts: List[int], k: int) -> int:
        a = sorted(gifts)
        for _ in range(k):
            top = a.pop()                      # lớn nhất
            insort(a, math.isqrt(top))         # chèn lại, giữ thứ tự tăng dần
        return sum(a)

In [13]:
sol = SolutionSorted()
print(sol.pickGifts([25, 64, 9, 4, 100], 4))     # 29
print(sol.pickGifts([1, 1, 1, 1], 4))            # 4

29
4


**Độ phức tạp:** Time `O(n log n + k * n)` (sắp xếp một lần; mỗi giây `pop` `O(1)` và `insort` `O(n)` do dịch phần tử), Space `O(n)`. Với `n <= 10^3` rất nhanh, và code dễ hiểu.

### Cách 4 — `heapreplace` + thoát sớm

Dùng `heapreplace` để thay đỉnh bằng giá trị mới trong một thao tác; dừng ngay khi đỉnh là `1`.

In [14]:
import heapq
import math
from typing import List

class SolutionReplace:
    def pickGifts(self, gifts: List[int], k: int) -> int:
        h = [-x for x in gifts]
        heapq.heapify(h)
        for _ in range(k):
            top = -h[0]
            if top == 1:                       # mọi đống đều là 1: không còn thay đổi
                break
            heapq.heapreplace(h, -math.isqrt(top))
        return -sum(h)

In [15]:
sol = SolutionReplace()
print(sol.pickGifts([25, 64, 9, 4, 100], 4))     # 29
print(sol.pickGifts([1, 1, 1, 1], 4))            # 4

29
4


**Độ phức tạp:** Time `O(n + min(k, T) log n)` với `T` là tổng số lần lấy căn cho đến khi mọi đống về `1` (không quá `~5n`), Space `O(n)`. Nhanh nhất khi `k` lớn.

## 8. TRACE từng bước

Trace Cách 4 (`heapreplace`) với `gifts = [25,64,9,4,100]`, `k = 4`. Heap lưu số đối nên nhìn "đỉnh" ta in `-h[0]`.

In [16]:
def trace_heap(gifts, k):
    h = [-x for x in gifts]
    heapq.heapify(h)
    print("gifts =", gifts, "| k =", k)
    for sec in range(1, k + 1):
        top = -h[0]
        if top == 1:
            print(f"  giây {sec}: đỉnh = 1 -> mọi đống đều là 1, dừng sớm")
            break
        new = math.isqrt(top)
        heapq.heapreplace(h, -new)
        print(f"  giây {sec}: lấy {top} -> để lại isqrt({top}) = {new}; các giá trị: {sorted(-x for x in h)}")
    total = -sum(h)
    print("  tổng còn lại:", total)
    return total

trace_heap([25, 64, 9, 4, 100], 4)
trace_heap([1, 1, 1, 1], 4)
trace_heap([2, 3], 5)

gifts = [25, 64, 9, 4, 100] | k = 4
  giây 1: lấy 100 -> để lại isqrt(100) = 10; các giá trị: [4, 9, 10, 25, 64]
  giây 2: lấy 64 -> để lại isqrt(64) = 8; các giá trị: [4, 8, 9, 10, 25]
  giây 3: lấy 25 -> để lại isqrt(25) = 5; các giá trị: [4, 5, 8, 9, 10]
  giây 4: lấy 10 -> để lại isqrt(10) = 3; các giá trị: [3, 4, 5, 8, 9]
  tổng còn lại: 29
gifts = [1, 1, 1, 1] | k = 4
  giây 1: đỉnh = 1 -> mọi đống đều là 1, dừng sớm
  tổng còn lại: 4
gifts = [2, 3] | k = 5
  giây 1: lấy 3 -> để lại isqrt(3) = 1; các giá trị: [1, 2]
  giây 2: lấy 2 -> để lại isqrt(2) = 1; các giá trị: [1, 1]
  giây 3: đỉnh = 1 -> mọi đống đều là 1, dừng sớm
  tổng còn lại: 2


2

## 9. Code nộp LeetCode

Max-heap bằng `heapq` (có thoát sớm):

In [17]:
import heapq
import math
from typing import List

class Solution:
    def pickGifts(self, gifts: List[int], k: int) -> int:
        h = [-x for x in gifts]
        heapq.heapify(h)
        for _ in range(k):
            top = -h[0]
            if top == 1:
                break
            heapq.heapreplace(h, -math.isqrt(top))
        return -sum(h)

In [18]:
print(Solution().pickGifts([25, 64, 9, 4, 100], 4))
print(Solution().pickGifts([1, 1, 1, 1], 4))

29
4


## 10. Vì sao thuật toán đúng

1. **Đúng thao tác.** Mỗi giây ta lấy ra giá trị lớn nhất `top` của đa tập các số đống quà và đặt vào `isqrt(top) = floor(sqrt(top))`. Đúng yêu cầu: để lại `floor(sqrt)` quà và lấy phần còn lại.
2. **Heap cho đúng đỉnh.** Heap lưu `-x`, nên phần tử nhỏ nhất `h[0]` ứng với `x` lớn nhất. Sau mỗi `heapreplace`, tính chất heap được khôi phục nên đỉnh luôn là lớn nhất của tập hiện tại.
3. **Đống bằng nhau.** Nếu nhiều đống cùng lớn nhất, chọn đống nào cũng đưa về cùng đa tập kết quả vì các đống đó giống hệt nhau.
4. **Thoát sớm hợp lệ.** Nếu lớn nhất là `1` thì mọi đống `>= 1` đều bằng `1`, và `isqrt(1) = 1` nên mỗi giây sau không đổi gì. Dừng sớm cho cùng tổng.
5. **Tổng.** Cuối cùng cộng mọi phần tử (đổi dấu) đúng là tổng quà còn lại.
6. **Các cách khác tương đương.** Cách `max/index` và cách `insort` lấy cùng đống lớn nhất mỗi giây (giá trị lớn nhất giống nhau), nên cho cùng kết quả; bộ test mục 12 xác nhận.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. `max` + `index` | `O(k * n)` | `O(n)` |
| 2. Max-heap `heappop`/`heappush` | `O(n + k log n)` | `O(n)` |
| 3. Mảng sắp xếp + `insort` | `O(n log n + k * n)` | `O(n)` |
| 4. `heapreplace` + thoát sớm | `O(n + min(k, T) log n)` | `O(n)` |

`n = len(gifts) <= 10^3`, `k <= 10^3`. Mỗi số lấy căn tối đa khoảng `5` lần là về `1`, nên số bước thực sự thay đổi không quá `5n`.

## 12. Bộ test

### 12.1. Ví dụ của đề, edge case và phản ví dụ

Các kỳ vọng được tính bằng một vòng mô phỏng độc lập (ghi chú trong cell).

In [19]:
sols = [SolutionBrute(), SolutionHeap(), SolutionSorted(), SolutionReplace(), Solution()]
cases = [
    ([25, 64, 9, 4, 100], 4, 29),        # ví dụ 1
    ([1, 1, 1, 1], 4, 4),                # ví dụ 2
    ([1], 1, 1),                         # một đống bằng 1
    ([4], 1, 2),                         # 4 -> 2
    ([10**9], 1, 31622),                 # biên trên một lần: isqrt(10^9) = 31622
    ([10**9], 2, 177),                   # 31622 -> 177
    ([9, 9], 1, 12),                     # hai đống bằng nhau, chọn một: 3 + 9
    ([9, 9], 2, 6),                      # 3 + 3
    ([2, 3], 3, 2),                      # 3->1, 2->1, rồi 1 giữ nguyên: 1 + 1
    ([100, 100, 100], 2, 120),           # 10 + 10 + 100
    ([16, 81, 1], 3, 8),                 # 81->9, 16->4, 9->3: [4, 3, 1]
    ([5, 5, 5], 5, 4),                   # ba lần 5->2, rồi hai lần 2->1: [1, 1, 2]
    ([3, 8, 15, 24], 1, 30),             # 24 -> 4: 3 + 8 + 15 + 4
    ([3, 8, 15, 24], 4, 10),
    ([10**9, 10**9], 3, 31799),          # 31622 + 177
    ([7], 1000, 1),                      # k lớn: về 1
    ([36, 49, 64], 1000, 3),             # mọi đống về 1
]
for gifts, k, want in cases:
    for s in sols:
        got = s.pickGifts(gifts[:], k)
        assert got == want, (type(s).__name__, gifts, k, got, want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 17


Phản ví dụ: với `gifts = [10]`, `k = 1`: `isqrt(10) = 3` nhưng `ceil(sqrt(10)) = 4`; hai cách làm tròn cho đáp án khác nhau, và đề yêu cầu **làm tròn xuống**.

In [20]:
import math
assert Solution().pickGifts([10], 1) == 3
assert math.ceil(math.sqrt(10)) == 4 != 3
print("phản ví dụ OK: floor(sqrt(10)) = 3, ceil = 4")

phản ví dụ OK: floor(sqrt(10)) = 3, ceil = 4


### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh dữ liệu đúng constraints: `1 <= n <= 1000`, `1 <= gifts[i] <= 10^9`, `1 <= k <= 1000`. Trộn nhiều kiểu: giá trị nhỏ (nhiều trùng), giá trị lớn tới `10^9`, và `k` nhỏ/lớn.

In [21]:
import random
random.seed(2558)

def gen():
    n = random.randint(1, random.choice([5, 20, 200, 1000]))
    hi = random.choice([3, 20, 1000, 10**6, 10**9])
    gifts = [random.randint(1, hi) for _ in range(n)]
    k = random.randint(1, random.choice([3, 30, 1000]))
    return gifts, k

def oracle(gifts, k):
    g = gifts[:]
    for _ in range(k):
        m = max(g)
        g[g.index(m)] = math.isqrt(m)
    return sum(g)

for t in range(1200):
    gifts, k = gen()
    assert 1 <= len(gifts) <= 1000 and 1 <= k <= 1000 and all(1 <= x <= 10**9 for x in gifts)
    ans = [s.pickGifts(gifts[:], k) for s in sols]
    assert len(set(ans)) == 1 and ans[0] == oracle(gifts, k), (len(gifts), k, ans)
print("1200 test ngẫu nhiên: 5 cách khớp nhau và khớp oracle")

1200 test ngẫu nhiên: 5 cách khớp nhau và khớp oracle


### 12.3. Tính chất: đơn điệu và không phá đầu vào

Tổng còn lại **không tăng** khi `k` tăng; và `gifts` đầu vào không bị thay đổi sau khi gọi (các cách `Brute`, `Heap`, `Sorted`, `Replace` đều làm việc trên bản sao).

In [22]:
random.seed(3)
for _ in range(300):
    gifts, _k = gen()
    before = gifts[:]
    prev = sum(gifts) + 1
    for k in (1, 2, 5, 20, 100):
        cur = Solution().pickGifts(gifts[:], k)
        assert cur <= prev
        prev = cur
    for s in sols:
        s.pickGifts(gifts, 10)
        assert gifts == before, type(s).__name__
print("tính chất đơn điệu theo k và không phá đầu vào đều đúng")

tính chất đơn điệu theo k và không phá đầu vào đều đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `ceil` thay vì làm tròn xuống

In [23]:
import math
def sai_ceil(gifts, k):
    g = gifts[:]
    for _ in range(k):
        i = g.index(max(g))
        g[i] = math.ceil(math.sqrt(g[i]))                  # SAI: đề yêu cầu floor
    return sum(g)

print("đúng:", Solution().pickGifts([10], 1), "| sai:", sai_ceil([10], 1))
assert Solution().pickGifts([10], 1) != sai_ceil([10], 1)

đúng: 3 | sai: 4


**Vì sao sai:** đề để lại `floor(sqrt(x))` quà. `math.isqrt` (hoặc `int(math.sqrt(x))`) làm tròn xuống; `ceil` làm tròn lên và cho đáp án lớn hơn khi `x` không phải số chính phương.

### Lỗi 2: Lấy căn **mọi** đống mỗi giây

In [24]:
def sai_moi_dong(gifts, k):
    g = gifts[:]
    for _ in range(k):
        g = [math.isqrt(x) for x in g]                      # SAI: chỉ đống lớn nhất bị lấy
    return sum(g)

gifts = [25, 64, 9, 4, 100]
print("đúng:", Solution().pickGifts(gifts, 1), "| sai:", sai_moi_dong(gifts, 1))
assert Solution().pickGifts(gifts, 1) != sai_moi_dong(gifts, 1)

đúng: 112 | sai: 28


**Vì sao sai:** mỗi giây chỉ **một** đống (đống nhiều quà nhất) bị thay đổi, không phải tất cả.

### Lỗi 3: Quên đẩy giá trị mới trở lại heap

In [25]:
def sai_quen_push(gifts, k):
    h = [-x for x in gifts]
    heapq.heapify(h)
    for _ in range(k):
        heapq.heappop(h)                                    # SAI: lấy ra rồi quên heappush(isqrt)
    return -sum(h)

print("đúng:", Solution().pickGifts([9, 9], 2), "| sai:", sai_quen_push([9, 9], 2))
assert Solution().pickGifts([9, 9], 2) != sai_quen_push([9, 9], 2)

đúng: 6 | sai: 0


**Vì sao sai:** đống bị chọn **vẫn còn** `isqrt(x)` quà. Nếu không đẩy lại, ta làm mất đống đó khỏi tổng (cũng mất nó khỏi các lần chọn sau).

### Lỗi 4: Chọn đống **nhỏ nhất** thay vì lớn nhất

In [26]:
def sai_nho_nhat(gifts, k):
    h = gifts[:]
    heapq.heapify(h)                                        # min-heap quên đổi dấu
    for _ in range(k):
        top = heapq.heappop(h)                              # SAI: đây là nhỏ nhất
        heapq.heappush(h, math.isqrt(top))
    return sum(h)

gifts = [25, 64, 9, 4, 100]
print("đúng:", Solution().pickGifts(gifts, 1), "| sai:", sai_nho_nhat(gifts, 1))
assert Solution().pickGifts(gifts, 1) != sai_nho_nhat(gifts, 1)

đúng: 112 | sai: 200


**Vì sao sai:** `heapq` là **min-heap**; muốn lấy lớn nhất phải lưu số đối `-x`. Ví dụ trên chọn `4` thay vì `100`.

### Lỗi 5: Làm hỏng đầu vào (sửa trực tiếp `gifts`)

In [27]:
def sai_pha_dau_vao(gifts, k):
    for _ in range(k):
        i = gifts.index(max(gifts))
        gifts[i] = math.isqrt(gifts[i])                     # SAI: sửa trực tiếp đầu vào của người gọi
    return sum(gifts)

gifts = [25, 64, 9, 4, 100]
before = gifts[:]
res = sai_pha_dau_vao(gifts, 4)
print("kết quả:", res, "| gifts sau khi gọi:", gifts)
assert res == 29 and gifts != before

kết quả: 29 | gifts sau khi gọi: [5, 8, 9, 4, 3]


**Cách tránh:** làm việc trên bản sao (`g = gifts[:]`) hoặc dùng heap dựng từ `gifts` (không sửa `gifts` gốc). Trên LeetCode việc sửa đầu vào thường chấp nhận, nhưng là thói quen xấu khi hàm bị gọi lại nhiều lần với cùng dữ liệu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lặp: lấy phần tử lớn nhất, biến đổi, đưa lại" ⇒ **heap / priority queue**.
- `heapq` chỉ có min-heap ⇒ lưu `-x` để có max-heap.
- `heapreplace(h, item)` gọn hơn `heappop` + `heappush`.
- "Làm tròn xuống căn bậc hai" ⇒ **`math.isqrt`**. Với `x <= 10^9`, `int(math.sqrt(x))` cũng đúng, nhưng `isqrt` chắc chắn, không lo sai số.
- Dãy lấy căn giảm cực nhanh (`10^9` về `1` sau 5 lần) ⇒ mô phỏng không thể lâu; có thể **thoát sớm** khi đỉnh là `1`.
- `n, k <= 10^3` nên cả cách `O(k * n)` cũng qua; chọn heap để luyện tập cho dữ liệu lớn hơn.

In [28]:
import random as _r
_r.seed(14)
for _ in range(20000):
    x = _r.randint(1, 10**9)
    assert int(math.sqrt(x)) == math.isqrt(x)
print("với x <= 10^9: int(math.sqrt(x)) trùng isqrt(x) trên 20000 giá trị thử; vẫn nên dùng isqrt cho chắc chắn")

với x <= 10^9: int(math.sqrt(x)) trùng isqrt(x) trên 20000 giá trị thử; vẫn nên dùng isqrt cho chắc chắn


## 15. Biến thể của bài

1. **Lấy phần tử nhỏ nhất** (min-heap): ví dụ "chọn đống ít quà nhất, đổi thành `floor(sqrt)`".
2. **Số giây ít nhất để mọi đống bằng `1`:** mô phỏng đến khi đỉnh heap bằng `1`.
3. **Thay vì `sqrt`, dùng `x // 2`** (LeetCode 1962 "Remove Stones to Minimize the Total": lấy `floor(x / 2)` khi chọn đống lớn nhất).
4. **Trả về mảng cuối cùng** thay vì tổng.
5. **Thao tác chọn `m` đống lớn nhất mỗi giây** (heap kích thước `m`).
6. **Phiên bản có nhiều lần truy vấn `k`** (dựng heap một lần, xử lý dần các thao tác).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `max` + `index` | quét tìm lớn nhất mỗi giây | `O(k * n)` | `O(n)` | oracle, dữ liệu nhỏ |
| 2. Max-heap | `heappop` + `heappush` | `O(n + k log n)` | `O(n)` | cách chuẩn khi `k`, `n` lớn |
| 3. Mảng sắp xếp + `insort` | lớn nhất ở cuối mảng | `O(n log n + k * n)` | `O(n)` | khi chưa quen heap |
| 4. `heapreplace` + thoát sớm | thay đỉnh một thao tác, dừng khi đỉnh bằng 1 | `O(n + min(k, T) log n)` | `O(n)` | cách nộp bài gọn và nhanh |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().pickGifts([4, 9], 2)` bằng bao nhiêu? Còn `Solution().pickGifts([16], 2)`?

**Bài 2 (tính tay).** Với `gifts = [49, 25, 100]` và `k = 3`, ghi từng giây và tổng cuối.

**Bài 3 (điền TODO).** Hoàn thành `seconds_until_all_ones(gifts)` trả về số giây cần để **mọi** đống đều bằng `1`.

In [29]:
def seconds_until_all_ones(gifts):
    h = [-x for x in gifts]
    heapq.heapify(h)
    secs = 0
    # TODO: trong khi đỉnh (-h[0]) > 1: heapreplace(h, -isqrt(đỉnh)); secs += 1
    return secs

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `remove_stones(piles, k)` (LeetCode 1962): mỗi lần chọn đống lớn nhất và bỏ đi `floor(x / 2)` viên (còn lại `x - floor(x / 2)` viên); trả về tổng còn lại sau `k` lần.

**Bài 5 (biến thể / mini challenge).** Viết `pick_gifts_min(gifts, k)`: mỗi giây chọn đống **ít quà nhất** (min-heap trực tiếp, không đổi dấu) và để lại `isqrt`. So sánh kết quả với `Solution().pickGifts` trên một vài mảng để thấy hai bài khác nhau.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[4,9]`: giây 1 lấy `9 -> 3` (`[4,3]`), giây 2 lấy `4 -> 2` (`[2,3]`), tổng `5`. `[16]`: `16 -> 4 -> 2`, tổng `2`.
- **Mức 2 (vừa):** Bài 2: giây 1: `100 -> 10` còn `[49,25,10]`; giây 2: `49 -> 7` còn `[7,25,10]`; giây 3: `25 -> 5` còn `[7,5,10]`; tổng `22`. Bài 3: vòng `while -h[0] > 1`.
- **Mức 3 (sâu):** Bài 4: với `x` lẻ ví dụ `5`, bỏ `floor(5/2) = 2` còn `3`. Bài 5: dùng `heappop`/`heappush` trực tiếp với giá trị dương.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert Solution().pickGifts([4, 9], 2) == 5
assert Solution().pickGifts([16], 2) == 2

# Bài 2
assert Solution().pickGifts([49, 25, 100], 3) == 22

# Bài 3
def seconds_until_all_ones(gifts):
    h = [-x for x in gifts]
    heapq.heapify(h)
    secs = 0
    while -h[0] > 1:
        heapq.heapreplace(h, -math.isqrt(-h[0]))
        secs += 1
    return secs

# Bài 4
def remove_stones(piles, k):
    h = [-x for x in piles]
    heapq.heapify(h)
    for _ in range(k):
        x = -h[0]
        heapq.heapreplace(h, -(x - x // 2))
    return -sum(h)

# Bài 5
def pick_gifts_min(gifts, k):
    h = gifts[:]
    heapq.heapify(h)
    for _ in range(k):
        x = heapq.heappop(h)
        heapq.heappush(h, math.isqrt(x))
    return sum(h)

assert seconds_until_all_ones([1, 1]) == 0
assert seconds_until_all_ones([4]) == 2           # 4 -> 2 -> 1
assert seconds_until_all_ones([10**9]) == 5       # 10^9 -> 31622 -> 177 -> 13 -> 3 -> 1
assert remove_stones([5, 4, 9], 2) == 12          # 9 -> 5, rồi 5 -> 3: [3, 4, 5]
assert remove_stones([4, 3, 6, 7], 3) == 12       # LeetCode 1962 ví dụ 2: đáp án 12
assert pick_gifts_min([25, 64, 9, 4, 100], 1) == 25 + 64 + 9 + 2 + 100
assert pick_gifts_min([25, 64, 9, 4, 100], 1) != Solution().pickGifts([25, 64, 9, 4, 100], 1)
random.seed(58)
for _ in range(100):
    g, k = gen()
    k = min(k, 50)
    assert seconds_until_all_ones(g) <= 5 * len(g)
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (lặp "lấy lớn nhất, biến đổi, đẩy lại" bằng max-heap)

```python
class Solution:
    def pickGifts(self, gifts: List[int], k: int) -> int:
        h = [-x for x in gifts]
        heapq.heapify(h)
        for _ in range(k):
            top = -h[0]
            if top == 1:
                break
            heapq.heapreplace(h, -math.isqrt(top))
        return -sum(h)
```

**Checklist**

- [ ] Lấy đống **lớn nhất** (heap lưu số đối) chứ không phải nhỏ nhất?
- [ ] Làm tròn **xuống** (`math.isqrt`), không dùng `ceil`/`round`?
- [ ] Chỉ **một** đống thay đổi mỗi giây, và đã đẩy giá trị mới lại heap?
- [ ] Trả về **tổng còn lại** (nhớ đổi dấu khi cộng heap số đối)?
- [ ] Đã thử `[25,64,9,4,100], k=4 -> 29` và `[1,1,1,1], k=4 -> 4`?
- [ ] Xử lý đỉnh bằng `1` (thoát sớm hoặc để `isqrt(1) = 1`)?
- [ ] Signature đúng `pickGifts(self, gifts: List[int], k: int) -> int` và có `import heapq, math`?